<a href="https://colab.research.google.com/github/MuXolotl/MuXVS/blob/arena/01a0c5ce-muxvs/assets/MuXVS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MuXVS — установка и запуск в Colab
- **Шаг 1** ставит всё и проверяет окружение, **шаг 2** запускает интерфейс.
- Нужен GPU: Среда выполнения → Сменить среду выполнения → T4 GPU.
- Обучаемые модели сохраняются на Google Drive (`MyDrive/MuXVS`), всё остальное — в папке репозитория.

Будьте в курсе обновлений: [Telegram-канал](https://t.me/politrees)

In [ ]:
# @title <big>⬇️ **ШАГ 1: Установка и проверка**
# @markdown Клонирует ветку, ставит зависимости, монтирует Drive и проверяет окружение.
REPO_URL = "https://github.com/MuXolotl/MuXVS"  # @param {type:"string"}
REPO_BRANCH = "arena/01a0c5ce-muxvs"  # @param {type:"string"}
ROOT_DIR = "/content/MuXVS"  # @param {type:"string"}

import os
import shutil
import subprocess
import sys

# --- GPU --- #
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Нет GPU! Среда выполнения → Сменить среду выполнения → T4 GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)} (torch {torch.__version__})")

# --- Drive (сюда сохраняются обучаемые модели) --- #
if not os.path.ismount("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
os.makedirs("/content/dataset", exist_ok=True)
print("Drive подключён.")

# --- Репозиторий --- #
if os.path.isdir(os.path.join(ROOT_DIR, ".git")):
    print(f"Репозиторий уже есть — обновляю ветку {REPO_BRANCH}...")
    for args in (
        ["fetch", "-q", "origin"],
        ["checkout", "-q", REPO_BRANCH],
        ["pull", "-q", "--ff-only", "origin", REPO_BRANCH],
    ):
        subprocess.run(["git", "-C", ROOT_DIR, *args], check=False)
else:
    print(f"Клонирую {REPO_URL} (ветка {REPO_BRANCH})...")
    subprocess.run(
        ["git", "clone", "-b", REPO_BRANCH, "--single-branch", REPO_URL, ROOT_DIR],
        check=True,
    )

# --- Зависимости (torch берётся встроенный в Colab) --- #
print("Ставлю зависимости, это займёт несколько минут...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(
    ["uv", "pip", "install", "--system", "-q",
     "-r", os.path.join(ROOT_DIR, "requirements.txt")],
    check=True,
)

# --- ffmpeg --- #
if shutil.which("ffmpeg") is None:
    print("Ставлю ffmpeg...")
    subprocess.run(["sudo", "apt-get", "update", "-q"], check=True)
    subprocess.run(["sudo", "apt-get", "install", "-y", "-q", "ffmpeg"], check=True)

shutil.rmtree("/content/sample_data", ignore_errors=True)
os.chdir(ROOT_DIR)
sys.path.insert(0, ROOT_DIR)

# --- Проверка --- #
import gradio as gr

print(f"\ngradio {gr.__version__}" + ("" if gr.__version__ == "5.35.0" else "  ⚠️ нужен 5.35.0!"))
print("ffmpeg:", shutil.which("ffmpeg") or "⚠️ НЕ НАЙДЕН")

from assets.env_paths import training_logs_dir

print("Модели обучения →", training_logs_dir())

from app import MuXVS

print(f"Интерфейс собирается: {len(MuXVS.blocks)} блоков.")
print("✅ Готово, запускайте ШАГ 2.")

In [ ]:
# @title <big>🚀 **ШАГ 2: Запуск интерфейса**
# @markdown ### **Способ запуска**:
launch_method = "Gradio (share)"  # @param ["Gradio (share)", "localtunnel", "cloudflared", "localhost.run", "Pinggy", "ngrok"]
# @markdown Стабильно работают без VPN:
# @markdown - localtunnel
# @markdown - localhost.run
# @markdown ---
# @markdown ### **Токен ngrok**: *(нужен только для ngrok: [ngrok.com](https://ngrok.com) → Your Authtoken)*
ngrok_token = ""  # @param {type:"string"}
# @markdown ### **Порт**:
PORT = 4000  # @param {type:"integer"}

ROOT_DIR = globals().get("ROOT_DIR", "/content/MuXVS")

import os
import re
import shutil
import socket
import subprocess
import sys
import time

if not os.path.isfile(os.path.join(ROOT_DIR, "app.py")):
    raise FileNotFoundError(f"В {ROOT_DIR} нет app.py — сначала запустите ШАГ 1.")
os.chdir(ROOT_DIR)

APP_LOG = os.path.join(ROOT_DIR, "app.log")
_OPEN_LOGS = []


def print_url(url):
    print("🔗 Ссылка для доступа в интерфейс:", url)


def cleanup():
    """Глушит прошлые запуски, чтобы порт был свободен."""
    for pattern in ("[a]pp.py", "[c]loudflared", "[l]t --port"):
        subprocess.run(["pkill", "-f", pattern], capture_output=True)
    time.sleep(2)


def wait_for_server(port, timeout=180):
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket() as sock:
            sock.settimeout(2)
            try:
                sock.connect(("127.0.0.1", port))
                return
            except OSError:
                time.sleep(2)
    print("❌ Сервер не поднялся за 3 минуты. Хвост лога:")
    subprocess.run(["tail", "-n", "60", APP_LOG])
    raise RuntimeError("Запуск не удался, см. app.log.")


def start_app(port):
    _OPEN_LOGS.append(open(APP_LOG, "w"))
    subprocess.Popen(
        [sys.executable, "app.py", "--port", str(port), "--no-share"],
        stdout=_OPEN_LOGS[-1],
        stderr=subprocess.STDOUT,
    )
    print(f"⏳ Жду сервер на порту {port}...")
    wait_for_server(port)
    print("✅ Сервер запущен.\n")


def ensure_ssh_key():
    key_path = os.path.expanduser("~/.ssh/id_rsa")
    if not os.path.exists(key_path):
        os.makedirs(os.path.dirname(key_path), exist_ok=True)
        subprocess.run(["ssh-keygen", "-t", "rsa", "-N", "", "-f", key_path, "-q"], check=True)


def ssh_tunnel_loop(ssh_args, url_patterns, reconnect_delay=5):
    url_res = [re.compile(p) for p in url_patterns]
    try:
        while True:
            proc = subprocess.Popen(
                ssh_args,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            link_printed = False
            try:
                for line in proc.stdout:
                    line = line.strip()
                    if not line or line.startswith("RB:"):
                        continue
                    if not link_printed:
                        for pattern in url_res:
                            m = pattern.search(line)
                            if m:
                                print_url(m.group(0).rstrip(").,"))
                                link_printed = True
                                break
            finally:
                proc.kill()
                proc.wait()
            print(f"\n⚠️ Соединение потеряно, переподключение через {reconnect_delay} сек...\n")
            time.sleep(reconnect_delay)
    except KeyboardInterrupt:
        print("\n⏹ Остановлено.")


def keep_alive():
    try:
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        print("\n⏹ Остановлено.")


print(f"Способ: {launch_method}, порт: {PORT}\n")

# ======== Gradio (share) ========
if launch_method == "Gradio (share)":
    print("⚠️ Туннель идёт через Cloudflare — у некоторых может не работать.")
    print("   Если не грузится — попробуйте localhost.run или localtunnel\n")
    cleanup()
    !python app.py --port {PORT}


# ======== localtunnel ========
elif launch_method == "localtunnel":
    if shutil.which("lt") is None:
        print("Ставлю localtunnel...")
        subprocess.run(["npm", "install", "-g", "localtunnel"], check=True)
    cleanup()
    start_app(PORT)

    ip_address = subprocess.check_output(["curl", "-s", "ifconfig.me"]).decode().strip()

    print("=" * 50)
    print("Пароль (IP-адрес) для доступа к ссылке ниже:")
    from IPython.display import display
    import ipywidgets as widgets

    display(widgets.Text(value=ip_address, description="Пароль (IP):", disabled=True))
    print("=" * 50 + "\n")

    !lt --port {PORT}


# ======== cloudflared ========
elif launch_method == "cloudflared":
    print("⚠️ Туннель идёт через Cloudflare — у некоторых может не работать.")
    print("   Если не грузится — попробуйте localhost.run или localtunnel\n")

    bin_path = "/tmp/cloudflared"
    if not os.path.isfile(bin_path):
        print("Скачиваю cloudflared...")
        subprocess.run(
            [
                "wget", "-q",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                "-O", bin_path,
            ],
            check=True,
        )
        os.chmod(bin_path, 0o755)

    cleanup()
    start_app(PORT)

    _OPEN_LOGS.append(open("cloudflared.log", "w"))
    subprocess.Popen(
        [bin_path, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=_OPEN_LOGS[-1],
        stderr=subprocess.STDOUT,
    )

    url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    public_url = None
    for _ in range(60):
        time.sleep(2)
        if os.path.exists("cloudflared.log"):
            with open("cloudflared.log") as f:
                m = url_re.search(f.read())
                if m:
                    public_url = m.group(0)
                    break

    if public_url:
        print_url(public_url)
    else:
        print("❌ Не удалось получить ссылку. Лог:")
        subprocess.run(["tail", "-n", "80", "cloudflared.log"])

    keep_alive()


# ======== localhost.run ========
elif launch_method == "localhost.run":
    cleanup()
    start_app(PORT)
    ensure_ssh_key()

    ssh_tunnel_loop(
        [
            "ssh",
            "-tt",
            "-o", "StrictHostKeyChecking=no",
            "-o", "UserKnownHostsFile=/dev/null",
            "-o", "ServerAliveInterval=30",
            "-o", "ServerAliveCountMax=3",
            "-o", "ConnectTimeout=15",
            "-R", f"80:127.0.0.1:{PORT}",
            "nokey@localhost.run",
        ],
        url_patterns=[
            r"https://[a-z0-9]+\.lhr\.life",
        ],
    )


# ======== Pinggy ========
elif launch_method == "Pinggy":
    cleanup()
    start_app(PORT)
    ensure_ssh_key()

    ssh_tunnel_loop(
        [
            "ssh",
            "-T",
            "-o", "StrictHostKeyChecking=no",
            "-o", "UserKnownHostsFile=/dev/null",
            "-o", "ServerAliveInterval=30",
            "-o", "ServerAliveCountMax=3",
            "-o", "ConnectTimeout=15",
            "-p", "443",
            f"-R0:127.0.0.1:{PORT}",
            "http@a.pinggy.io",
        ],
        url_patterns=[
            r"https://[a-z0-9-]+\.a\.free\.pinggy\.link",
            r"https://[a-z0-9-]+\.free\.pinggy\.link",
            r"https://[a-z0-9-]+\.a\.pinggy\.link",
            r"https://[a-z0-9-]+\.pinggy\.link",
        ],
    )


# ======== ngrok ========
elif launch_method == "ngrok":
    if not ngrok_token:
        raise ValueError("Для ngrok нужен токен: ngrok.com → Your Authtoken")
    try:
        from pyngrok import ngrok as ngrok_module
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
        from pyngrok import ngrok as ngrok_module

    ngrok_module.set_auth_token(ngrok_token)
    cleanup()
    start_app(PORT)

    public_url = ngrok_module.connect(PORT, bind_tls=True)
    print_url(public_url)

    try:
        keep_alive()
    finally:
        ngrok_module.disconnect(public_url)